In [1]:
import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter
from problems.ThickenShell import ThickenShell
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell
from Training.MainTrain import NN_Trainer, TrainingConfig
from Decoder_CLasses.ContinuousVoronoiDecoder import ContinuousVoronoiDecoder
from Decoder_CLasses.NearUniformHoneycombBaseline import NearUniformHoneycombBaseline
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import IntrinsicSurfaceHoneycombBaseline
from Pred_NN_Classes.ppnet import PPNet
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d, cKDTree,QhullError
from Utils.ExportAbaqus_VoxelBased import export_abaqus_voxel_fem
from Utils.ExportAbaqus import export_abaqus_shell
from Utils.embedded_centroid_stress_audit import (
    export_embedded_centroid_stress_audit,
    print_stress_case_summary,
)
from Utils.RunFEM_ExportVoxelABA import VoxelFEMAbaqusRunner
from Decoder_CLasses import Phase1VoronoiDecoder

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)

%matplotlib inline
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

Code Directory: /home/arash/HVD_SeedsBase
Test Step files Directory: /home/arash/HVD_SeedsBase/Testparts
device: cuda
PyVista backend: trame


In [2]:

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_surface_density(fields, density_key="rho"):
    face_tensor = fields["face_tensor"]

    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary.
    if triangles.min() == 1:
        triangles = triangles - 1

    # PyVista face format:
    # [3, i, j, k, 3, i, j, k, ...]
    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    mesh = pv.PolyData(points, pv_faces)

    density = to_numpy(
        fields[density_key]
    ).reshape(-1)

    mesh.point_data["density"] = density

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": density_key,
        },
    )

    plotter.add_text(
        f"Surface density: {density_key}",
        font_size=12,
    )

    plotter.show_axes()
    plotter.show()

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv


def show_fem_distributions(
    verification,
    *,
    case_name,
    stress_limit=None,
    bins=30,
    show_edges=False,
    output_dir="FEM_Maps",
    clim_stress=None,
    clim_displacement=None,
):
    fe = verification.embedded_fem.fe
    mesh = fe.mesh

    # Stress: maximum of 8 integration points per active element
    stress = (
        fe.stress_vm_ip_active
        .detach()
        .cpu()
        .numpy()
        .max(axis=1)
    )
    element_ids = np.asarray(mesh.active_element_ids, dtype=np.int64)

    # Displacement magnitude: one value per mesh node
    u = (
        fe.u.detach()
        .cpu()
        .numpy()
        .reshape(mesh.numNodes, 3)
    )
    displacement = np.linalg.norm(u, axis=1)
    active_node_ids = np.asarray(mesh.active_node_ids, dtype=np.int64)

    if len(stress) != len(element_ids):
        raise ValueError("Stress values do not match the active elements.")

    # Build a grid containing active C3D8 elements only
    connectivity = mesh.elemNodes[element_ids].astype(np.int64)
    cells = np.column_stack([
        np.full(len(element_ids), 8, dtype=np.int64),
        connectivity,
    ]).ravel()

    grid = pv.UnstructuredGrid(
        cells,
        np.full(len(element_ids), pv.CellType.HEXAHEDRON, dtype=np.uint8),
        np.asarray(mesh.nodeXYZ),
    )

    stress_name = "Max IP von Mises (MPa)"
    displacement_name = "Displacement magnitude (mm)"

    grid.cell_data[stress_name] = stress
    grid.point_data[displacement_name] = displacement
    surface = grid.extract_surface()

    # Two maps, with the same camera position
    plotter = pv.Plotter(shape=(1, 2), window_size=(1600, 700))

    Clim_stress_Def = None

    stress_range = (
        (0.0, float(stress.max()))
        if clim_stress is None
        else tuple(clim_stress)
    )

    displacement_range = (
        (0.0, float(displacement[active_node_ids].max()))
        if clim_displacement is None
        else tuple(clim_displacement)
    )

    plotter.subplot(0, 0)
    plotter.add_mesh(
        surface,
        scalars=stress_name,
        preference="cell",
        cmap="turbo",
        clim=stress_range,
        show_edges=show_edges,
        scalar_bar_args={"title": stress_name},
    )
    plotter.add_text(f"{case_name}: stress", font_size=11)
    plotter.add_axes()

    plotter.subplot(0, 1)
    plotter.add_mesh(
        surface,
        scalars=displacement_name,
        preference="point",
        cmap="viridis",
        clim=displacement_range,
        show_edges=show_edges,
        scalar_bar_args={"title": displacement_name},
    )
    plotter.add_text(f"{case_name}: displacement", font_size=11)
    plotter.add_axes()

    plotter.link_views()
    plotter.reset_camera()

    # Histograms: one count per active element and per active node
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(stress, bins=bins)
    if stress_limit is not None:
        axes[0].axvline(
            stress_limit,
            color="red",
            linestyle="--",
            label=f"Limit: {stress_limit:g} MPa",
        )
        axes[0].legend()
    axes[0].set_xlabel(stress_name)
    axes[0].set_ylabel("Number of active elements")
    axes[0].set_title("Element stress distribution")

    axes[1].hist(displacement[active_node_ids], bins=bins)
    axes[1].set_xlabel(displacement_name)
    axes[1].set_ylabel("Number of active nodes")
    axes[1].set_title("Nodal displacement distribution")

    fig.suptitle(case_name)
    fig.tight_layout()

    # Save each model under its own name
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    grid.save(output_dir / f"{case_name}_fields.vtu")
    fig.savefig(output_dir / f"{case_name}_histograms.png", dpi=200)

    plotter.show(screenshot=str(output_dir / f"{case_name}_maps.png"))
    plt.show()

    worst_element_rank = int(np.argmax(stress))
    worst_node = int(active_node_ids[np.argmax(displacement[active_node_ids])])

    print(f"Maximum stress: {stress.max():.4f} MPa")
    print(f"  Element centre: {mesh.elemCenters[element_ids[worst_element_rank]]}")
    print(f"Maximum displacement: {displacement[worst_node]:.6f} mm")
    print(f"  Node position: {mesh.nodeXYZ[worst_node]}")

    if stress_limit is not None:
        above = int(np.count_nonzero(stress > stress_limit))
        print(
            f"Elements above {stress_limit:g} MPa: "
            f"{above}/{len(stress)} ({100 * above / len(stress):.2f}%)"
        )

    return grid

In [3]:
def show_two_surface_densities(
    face_tensor,
    density_phase1,
    density_phase2,
    *,
    title_phase1="Phase 1 - Continuous",
    title_phase2="Phase 2 - Graph",
    binary=False,
    threshold=0.5,
):
    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary
    if triangles.min() == 1:
        triangles = triangles - 1

    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    # --------------------------------------------------------
    # Densities
    # --------------------------------------------------------

    rho1 = to_numpy(density_phase1).reshape(-1)
    rho2 = to_numpy(density_phase2).reshape(-1)

    if len(rho1) != len(points):
        raise ValueError(
            f"Phase-1 density has {len(rho1)} values "
            f"but surface has {len(points)} vertices."
        )

    if len(rho2) != len(points):
        raise ValueError(
            f"Phase-2 density has {len(rho2)} values "
            f"but surface has {len(points)} vertices."
        )

    if binary:
        rho1 = (rho1 > threshold).astype(float)
        rho2 = (rho2 > threshold).astype(float)

    # --------------------------------------------------------
    # Separate meshes so the scalar fields do not overwrite
    # each other
    # --------------------------------------------------------

    mesh1 = pv.PolyData(points, pv_faces)
    mesh2 = pv.PolyData(points, pv_faces)

    mesh1.point_data["density"] = rho1
    mesh2.point_data["density"] = rho2

    # --------------------------------------------------------
    # Side-by-side visualization
    # --------------------------------------------------------

    plotter = pv.Plotter(
        shape=(1, 2),
        window_size=(1700, 750),
    )

    # Phase 1
    plotter.subplot(0, 0)

    plotter.add_mesh(
        mesh1,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase1,
        font_size=12,
    )

    plotter.add_axes()

    # Phase 2
    plotter.subplot(0, 1)

    plotter.add_mesh(
        mesh2,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase2,
        font_size=12,
    )

    plotter.add_axes()

    # Same camera for both
    plotter.link_views()
    plotter.reset_camera()

    plotter.show()

    return mesh1, mesh2

In [4]:
viz = CADVisualizer()
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Planar = TesPartsDir / "Planar.stp"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Helmet = TesPartsDir / "Helmet_New.stp"
FreeForm_Bend = TesPartsDir / "FreeForm_bend.stp"
CircBracket_Half = TesPartsDir / "CircBracket_Half.stp"
HelmetNew_Half = TesPartsDir / "HelmetNew_Half.stp"
Wing_NACA2412 = TesPartsDir / "Wing_NACA2412.stp"

aircraft_radome = TesPartsDir / "aircraft_radome.stp"

shape_path = Planar
Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale = 0.2,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(shape_path)
dx,dy,dz = Face_Cad.print_face_info();

points = face_mesh["points_xyz"].detach().cpu().numpy()
faces = face_mesh["pv_faces"].detach().cpu().numpy()

mesh = pv.PolyData(points, faces)

plotter = pv.Plotter()
plotter.add_mesh(
    mesh,
    show_edges=True,
)
plotter.add_axes()
plotter.show()

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/Document' (2D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 30%] Meshing curve 2 (BSpline)
Info    : [ 60%] Meshing curve 3 (BSpline)
Info    : [ 80%] Meshing curve 4 (BSpline)
Info    : Done meshing 1D (Wall 0.000594905s, CPU 0.000626s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 1.68375s, CPU 1.6538s)
Info    : 79739 nodes 159480 elements
Info    : Clearing all models and views...
Info    : Done clearing all models and views

=== Active Face Info ===
XYZ bounds:
  X: [-0.000000, 10.000000]  span=10.000000
  Y: [-0.000000, 10.000000]  span=10.000000
  Z: [-0.000000, 0.000000]  span=0.000000

UV bounds:
  U: [-10.000000, 0.000000]
  V: [0.000000, 10.000000]

Periodic:
  U periodic: False
  V periodic: False



Widget(value='<iframe src="http://localhost:41283/index.html?ui=P_0x72825b935f30_0&reconnect=auto" class="pyvi…

In [6]:
Strut_Thickness = 0.4
decoder = ContinuousVoronoiDecoder(
    Cad_domain=Face_Cad,
    face_mesh=face_mesh,
    n_seeds=100,
    strut_thickness=Strut_Thickness,
    seed_domain_margin=0.8,
    # Keep your other settings unchanged.
)
baseline = NearUniformHoneycombBaseline(
    decoder=decoder,
    cad_domain=Face_Cad,
    face_mesh=face_mesh,

    target_seed_count=None,
    spacing_uv=0.2,

    mode="uv_hex",
    relaxation_steps=0,
    relaxation_factor=0.5,

    include_exterior_support_sites=True,
    support_band_cells=2.5,

    relax_support_sites=True,
    support_relaxation_neighbors=6,
    max_relaxation_displacement_cells=0.75,
    preserve_lattice_topology=True,
)

honeycomb_fields = baseline.generate(
    device=device,
    dtype=torch.float64,
    generate_density_fiber=True,
)

curves_uv = honeycomb_fields["edge_curves_uv"]          # [E, K, 2]
curves_xyz = honeycomb_fields["edge_curves_xyz"]        # [E, K, 3]
graph = honeycomb_fields["graph"]
honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"] 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(honeycomb_fields, density_key="rho")





total all curve length: 145.4253489085009
total VD curve length: 105.42534781546871


Widget(value='<iframe src="http://localhost:41283/index.html?ui=P_0x72825b935030_2&reconnect=auto" class="pyvi…

In [ ]:
voxel_size = float((dx + dy + dz) / 200.0)
thickness = 3.0 * voxel_size
print(f"dx:{dx} ,dy:{dy} ,dz:{dz}")
print(f"voxel size : {voxel_size}")
print(f"thickness : {thickness}")

In [ ]:
common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    faces_index_base=0,
)

shell_problem = ThickenShell(
    **common_settings
)

In [ ]:
shell_problem.show_boundaries(
    segments_per_curve=2
)

In [ ]:
# test_loads = [

#     {
#         "type": "force",
#         "boundaries": [5, 6, 7],
#         "magnitude": 50.0,
#         "direction": "-z",
#     },

#     {
#         "type": "distributed_force",
#         "boundaries": [17, 18, 19],
#         "magnitude": 100.0,
#         "direction": "y",
#         "distribution": "linear_decreasing",
#     },

#     {
#         "type": "moment",
#         "boundaries": [13, 14, 15],
#         "magnitude": 25.0,
#         "direction": "x",
#     },
# ]

In [ ]:
Boundary_loads = [

    {
        "type": "force",
        "boundaries": [2,3],
        "magnitude": 1,
        "direction": "-z",
    },
    {
        "type": "distributed_force",
        "boundaries": [0,1],
        "magnitude": 5.0,
        "direction": "-y",
        "distribution": "parabolic",
    },
]

In [ ]:
BC_Report = shell_problem.boundary_loading(
    fixed=[6,7],
    loads=Boundary_loads,
    band=0.05* voxel_size,
)

In [ ]:
# Sueface_loads = [

    # # Axial force
    # {
    #     "type": "force",
    #     "axis": "x",
    #     "side": "max",
    #     "magnitude": 5.0,
    #     "direction": "x",
    # },

    # # Downward transverse force
    # {
    #     "type": "force",
    #     "axis": "x",
    #     "side": "max",
    #     "magnitude": 3.0,
    #     "direction": "-z",
    # },

    # Bending moment
    # {
    #     "type": "moment",
    #     "axis": "x",
    #     "side": "max",
    #     "magnitude": 20.0,
    #     "direction": "z",
    # },

    # # Torsional moment
    # {
    #     "type": "moment",
    #     "axis": "x",
    #     "side": "max",
    #     "magnitude": 50.0,
    #     "direction": "x",
    # },
# ]

In [ ]:
# BC_Report = shell_problem.surface_loading(
#     fixed_axis="x",
#     fixed_side="min",

#     loads=Sueface_loads,

#     fixed_band=0.002 * voxel_size,
#     load_band=0.001 * voxel_size,
# )

In [ ]:
voxel_size = float((dx + dy + dz) / 220.0)
thickness = 3.0 * voxel_size
print(f"dx:{dx} ,dy:{dy} ,dz:{dz}")
print(f"voxel size : {voxel_size}")
print(f"thickness : {thickness}")
LoadingCase ="ReaLCase"
common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    faces_index_base=0,
)

shell_problem = ThickenShell(
    **common_settings
)
Boundary_loads = [

    {
        "type": "force",
        "boundaries": [2,3],
        "magnitude": 1,
        "direction": "z",
    },
    {
        "type": "distributed_force",
        "boundaries": [0,1],
        "magnitude": 5.0,
        "direction": "y",
        "distribution": "parabolic",
    },
]

shell_problem.show_boundaries(
    segments_per_curve=2
)
BC_Report = shell_problem.boundary_loading(
    fixed=[6,7],
    loads=Boundary_loads,
    band=0.05* voxel_size,
)

In [ ]:
loading_img = shell_problem.show_voxels_surface_and_bc(
    return_img=True,
    off_screen=True,
    window_size=(560, 430),
    show=True,
)

In [ ]:
shell_problem.show_boundaries(
    segments_per_curve=1
)

In [ ]:

# LoadingCase = "tensile_compression"
# common_settings = dict(
#     thickness=thickness,
#     voxel_size=voxel_size,
#     extra_layers=1,
#     tensors=face_mesh,

#     voxelization_mode="triangle_distance",
#     subvoxel_samples=1,
#     min_geom_fraction=0.25,

#     bc_mapping_mode="surface_patch",
#     faces_index_base=0,
# )
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="fixed_side_loading",

#     # X identifies the two ends of the specimen.
#     BC_dir="x",

#     # Clamp the X-maximum end and load the X-minimum end.
#     fixed_side="min",
#     force_side="max",

#     # Apply the force along the global Z-axis.
#     load_dir="x",

#     # "max" means positive Z.
#     load_direction_side="max",

#     # Total applied force = +5 N.
#     Load_magnitude=0.000001,

#     # Select only outward-facing voxel faces at the X-maximum end.
#     fixed_region={
#         "extent_axis": "x",
#         "extent_side": "min",
#         "band": 0.001 * voxel_size,
#         "face_axis": "x",
#         "face_side": "min",
#     },

#     # Select only outward-facing voxel faces at the X-minimum end.
#     load_region={
#         "extent_axis": "x",
#         "extent_side": "max",
#         "band": 0.0005 * voxel_size,
#         "face_axis": "x",
#         "face_side": "max",
#         "direction": "z",
#         "total_force": -1,
#     },

# )
# loading_img = shell_problem.show_voxels_surface_and_bc(
#     return_img=True,
#     off_screen=True,
#     window_size=(560, 430),
#     show=True,
# )

In [ ]:
verification = VoxelFEMAbaqusRunner(
    face_mesh=face_mesh,
    shell_problem=shell_problem,
    device=device,
    output_dir="FEM_Verification",
)
LoadingCase= "bend"

In [ ]:
Strut_Thickness = 0.4
decoder = ContinuousVoronoiDecoder(
    Cad_domain=Face_Cad,
    face_mesh=face_mesh,
    n_seeds=100,
    strut_thickness=Strut_Thickness,
    seed_domain_margin=0.8,
    # Keep your other settings unchanged.
)
baseline = NearUniformHoneycombBaseline(
    decoder=decoder,
    cad_domain=Face_Cad,
    face_mesh=face_mesh,

    target_seed_count=None,
    spacing_uv=0.170,

    mode="uv_hex",
    relaxation_steps=0,
    relaxation_factor=0.5,

    include_exterior_support_sites=True,
    support_band_cells=2.5,

    relax_support_sites=True,
    support_relaxation_neighbors=6,
    max_relaxation_displacement_cells=0.75,
    preserve_lattice_topology=True,
)

honeycomb_fields = baseline.generate(
    device=device,
    dtype=torch.float64,
    generate_density_fiber=True,
)

curves_uv = honeycomb_fields["edge_curves_uv"]          # [E, K, 2]
curves_xyz = honeycomb_fields["edge_curves_xyz"]        # [E, K, 3]
graph = honeycomb_fields["graph"]
honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"] 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(honeycomb_fields, density_key="rho")





In [ ]:
#honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
optimized_metrics = verification.run_fields(
    fields=honeycomb_fields,
    case_name="LL_Test" + shape_path.stem + "-Honeycomb-" + LoadingCase + f"_St{str(Strut_Thickness)[2:]}",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)
edge_lengths = honeycomb_fields["edge_curve_lengths_xyz"]          # [E]
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"]  
limit =300        # scalar tensor
stress = verification.embedded_fem.fe.stress_vm_ip_active.detach().cpu().numpy().max(axis=1)
print("Above limit:", np.count_nonzero(stress > limit))
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))

show_surface_density(honeycomb_fields, density_key="rho")
grid = show_fem_distributions(
    verification,
    case_name="HoneyComb",
    stress_limit=300.0,
    bins=5,
    show_edges=True,
    clim_stress=(0, 300),       # MPa
    #clim_displacement=(0, 0.5),  # mm
)

In [ ]:
from Training.MainTrain import (
    _field_to_numpy,
    load_optimized_shell_function,
    evaluate_optimized_shell_function,
    visualize_optimized_shell_fields,
)

Path1 = "/home/arash/HVD_SeedsBase/Case_Studies/Wing_NACA2412_50_Sth4_ReaLCase/optimized_shell_function.pt"
path2 = "/home/arash/HVD_SeedsBase/Case_Studies/PlanarPlannar_Seeds-70_Sth033/BestFeasibleCheckpoint/optimized_shell_function.pt"

Mpath =Path1
if not os.path.exists(Mpath):
    raise FileNotFoundError(f"Optimized shell function file not found: {Mpath}")
opt_fn = load_optimized_shell_function(Mpath, device="cpu")
fields = evaluate_optimized_shell_function(opt_fn)
fields["rho"] = (fields["rho"]>0.5).to(torch.float64)
optimized_metrics = verification.run_fields(
    fields=fields,
    case_name=shape_path.stem + "-OptimizedModel_Bend_St0174_F5_V80_St025",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)

total_all = fields["total_curve_length"]                 # scalar tensor
total_vd = fields["total_voronoi_curve_length"]
stress = verification.embedded_fem.fe.stress_vm_ip_active.detach().cpu().numpy().max(axis=1)
print("Above limit:", np.count_nonzero(stress > limit)) 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(fields, density_key="rho")
grid = show_fem_distributions(
    verification,
    case_name="Optimized",
    stress_limit=300.0,
    bins=5,
    show_edges=True,
    clim_stress=(0, 300),       # MPa
    #clim_displacement=(0, 0.5),  # mm
)

In [2]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 72718), started 2 days, 5:17:43 ago. (Use '!kill 72718' to kill it.)